# CLIP Token Map

Embed the CLIP vocabulary and reduce it to a searchable two-dimensional table. Start with the small sample; set `limit = None` for the full vocabulary.

In [ ]:
import csv
from pathlib import Path
import matplotlib.pyplot as plt
import torch
import umap
from transformers import CLIPModel, CLIPTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "openai/clip-vit-base-patch32"
tokenizer = CLIPTokenizer.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(device).eval()

In [ ]:
limit = 1000
vocabulary = sorted(tokenizer.get_vocab().items(), key=lambda item: item[1])
if limit is not None:
    vocabulary = vocabulary[:limit]
tokens, token_ids = zip(*vocabulary, strict=True)
clean_tokens = [tokenizer.decode([token_id]) for token_id in token_ids]
inputs = tokenizer(clean_tokens, return_tensors="pt", padding=True, truncation=True).to(device)
with torch.inference_mode():
    embeddings = model.get_text_features(**inputs)
    embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
len(vocabulary), embeddings.shape

In [ ]:
points = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42).fit_transform(embeddings.cpu().numpy())
fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(points[:, 0], points[:, 1], s=3, alpha=0.5)
ax.set(title=f"UMAP of {len(vocabulary):,} CLIP tokens", xticks=[], yticks=[]);

In [ ]:
output = Path("../outputs/token_map/notebook_umap.csv")
output.parent.mkdir(parents=True, exist_ok=True)
with output.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["token_id", "token", "clean_token", "x", "y"])
    writer.writerows((token_id, token, clean, x, y) for (token, token_id), clean, (x, y) in zip(vocabulary, clean_tokens, points, strict=True))
output